# [9665] FastText 2
Data file:
* https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/TEDx_talks.csv

In [ ]:
from datetime import datetime
print(f'Run time: {datetime.now().strftime("%D %T")}')

Run time: 04/01/25 20:22:09


### Import libraries

In [ ]:
import numpy as np
import pandas as pd
import string
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from gensim.models import FastText
from sklearn.metrics.pairwise import cosine_similarity
import logging

In [ ]:
%%time

import spacy

2025-04-01 20:22:19.082334: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


CPU times: user 8.82 s, sys: 1.25 s, total: 10.1 s
Wall time: 11 s


In [ ]:
# Load spaCy for lemmatization
nlp = spacy.load('en_core_web_sm')

In [ ]:
nltk.download('stopwords')
nltk.download('punkt')

[nltk_data] Downloading package stopwords to /Users/vj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to /Users/vj/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
# Setup loggimg
logging.basicConfig(format='%(asctime)s : %(levelname)s : %(message)s', level=logging.INFO)

### Load data

In [ ]:
df = pd.read_csv('https://raw.githubusercontent.com/vjavaly/Baruch-CIS-9665/main/data/TEDx_talks.csv')
df.shape

(4467, 7)

### Examine data

In [ ]:
df.sample(5)

,idx,main_speaker,title,details,posted,url,num_views
3420,b0041f3e8bb46c18bb104288f1f75efb,Hasan Elahi,"FBI, here I am!","After he ended up on a watch list by accident,...",Posted Oct 2011,https://www.ted.com/talks/hasan_elahi_fbi_here...,NaN
389,60b70dcc47f6fc0a5cef7705b5dce55c,Sergio Feferovich,La música de las ideas,From Mozart's operatic arias of love to João G...,Posted Aug 2019,https://www.ted.com/talks/sergio_feferovich_la...,NaN
2261,eea1057da9eff76332d2af5f8848eb6a,Nassim Assefi and Brian A. Levine,How in vitro fertilization (IVF) works,Infertility affects 1 in 8 couples worldwide. ...,Posted May 2015,https://www.ted.com/talks/nassim_assefi_and_br...,NaN
702,127e72e92587bad3b87de23420905567,Laura Bates,Everyday sexism,Most women experience sexism and harassment on...,Posted Feb 2019,https://www.ted.com/talks/laura_bates_everyday...,NaN
3734,d45f90c56338ca400df89d6281b78aa0,Dimitar Sasselov,How we found hundreds of potential Earth-like ...,"(NOTE: This talk was given in 2010, and this f...",Posted Jul 2010,https://www.ted.com/talks/dimitar_sasselov_how...,NaN


In [ ]:
pd.set_option('display.max_colwidth', None)

In [ ]:
df['details'].sample(5)

2868                                                                                                                                                                                                                                                 There's no actual law against women driving in Saudi Arabia. But it's forbidden. Two years ago, Manal al-Sharif decided to encourage women to drive by doing so -- and filming herself for YouTube. Hear her story of what happened next.
3579                                                                                                                                        Columnist David Brooks unpacks new insights into human nature from the cognitive sciences -- insights with massive implications for economics and politics as well as our own self-knowledge. In a talk full of humor, he shows how you can't hope to understand humans as separate individuals making choices based on their conscious awareness.
385     For the past 20 years, photographe

In [ ]:
df['main_speaker'] = df['main_speaker'].apply(lambda x: str(x).replace(" ", "") if isinstance(x, str) else x)
df['main_speaker'].sample(5)

1188      ZacharyR.Wood
3646         BirkeBaehr
4455    CameronSinclair
1087       ElizabethCox
2200       RichBenjamin
Name: main_speaker, dtype: object

### Prepare data

In [ ]:
# Clean up any NaN values in the dataset
df = df.fillna('')

In [ ]:
# Combine appropriate columns for model training
df['combined_text'] = df['main_speaker'] + ' ' + df['title'] + ' ' + df['details']
df['combined_text'].sample(5)

872     BrianD.Avery How rollercoasters affect your body In 1895, crowds flooded Coney Island to see America's first-ever looping coaster: the Flip Flap Railway. But its thrilling flip caused cases of severe whiplash, neck injury and even ejections. Today, coasters can pull off far more exciting tricks and do it safely. Brian D. Avery investigates what roller coasters are doing to your body and how they've managed to get scarier and safer at the same time. [TED-Ed Animation by Stretch Films Inc].
4426                                                                                                                                                                                                                                                                                                       Bono My wish: Three actions for Africa Musician and activist Bono accepts the 2005 TED Prize with a riveting talk, arguing that aid to Africa isn't just another celebrity cause; it's a global emergen

### Create function to preprocess text

In [ ]:
# Preprocessing: Tokenization, Stopword Removal, Lemmatization, and Punctuation Removal
def preprocess(text):
    # Check if text is a valid string
    if isinstance(text, str):
        # Remove all punctuation
        text = text.translate(str.maketrans('', '', string.punctuation))

        # Tokenize the text
        tokens = word_tokenize(text.lower())

        # Remove stopwords
        stop_words = set(stopwords.words('english'))
        tokens = [word for word in tokens if word not in stop_words]

        # Lemmatization using spaCy
        lemmatized_tokens = [token.lemma_ for token in nlp(" ".join(tokens)).doc]

        return lemmatized_tokens
    else:
        return []  # Return an empty list if the text is not a valid string

In [ ]:
%%time

# Clean combined columns field
df['combined_text_clean'] = df['combined_text'].astype(str).apply(preprocess)

CPU times: user 43.3 s, sys: 1.29 s, total: 44.6 s
Wall time: 49 s


In [ ]:
df['combined_text_clean'].sample(5)

1400                                                                                                                  [jonathankoch, datum, translation, toolkit, anyone, use, every, day, inundate, heap, datum, easy, way, decipher, easy, feel, overwhelmed, face, avalanche, information, three, simple, tool, jonathan, koch, outline, we, well, understand, datum, make, informed, decision, professional, personal, life]
3406                                                                                                                                                                                                                           [alexandertsiaras, conception, birth, —, visualize, imagemaker, alexander, tsiaras, share, powerful, medical, visualization, show, human, development, conception, birth, beyond, graphic, image]
2453                                                                                                                                                                  

### Train FastText model with cleaned data

In [ ]:
%%time

# Train the FastText model with the preprocessed data
model = FastText(sentences=df['combined_text_clean'], vector_size=100,
                 window=5, min_count=5, epochs=50)

2025-04-01 20:23:13,003 : INFO : collecting all words and their counts
2025-04-01 20:23:13,004 : INFO : PROGRESS: at sentence #0, processed 0 words, keeping 0 word types
2025-04-01 20:23:13,075 : INFO : collected 24801 word types from a corpus of 176448 raw words and 4467 sentences
2025-04-01 20:23:13,077 : INFO : Creating a fresh vocabulary
2025-04-01 20:23:13,122 : INFO : FastText lifecycle event {'msg': 'effective_min_count=5 retains 4691 unique words (18.91% of original 24801, drops 20110)', 'datetime': '2025-04-01T20:23:13.122281', 'gensim': '4.3.2', 'python': '3.8.8 (default, Apr 13 2021, 12:59:45) \n[Clang 10.0.0 ]', 'platform': 'macOS-10.16-x86_64-i386-64bit', 'event': 'prepare_vocab'}
2025-04-01 20:23:13,123 : INFO : FastText lifecycle event {'msg': 'effective_min_count=5 leaves 146395 word corpus (82.97% of original 176448, drops 30053)', 'datetime': '2025-04-01T20:23:13.123971', 'gensim': '4.3.2', 'python': '3.8.8 (default, Apr 13 2021, 12:59:45) \n[Clang 10.0.0 ]', 'platfor

CPU times: user 1min 11s, sys: 1.76 s, total: 1min 13s
Wall time: 31.3 s


### Use the trained model's most_similar method

In [ ]:
model.wv.most_similar('africa')

[('africas', 0.9715596437454224),
 ('african', 0.9182214736938477),
 ('hurricane', 0.6109551787376404),
 ('america', 0.604133665561676),
 ('americas', 0.5640692710876465),
 ('afraid', 0.5438988208770752),
 ('american', 0.5363386869430542),
 ('americans', 0.531467854976654),
 ('monica', 0.5271100401878357),
 ('subsaharan', 0.508916437625885)]

In [ ]:
model.wv.most_similar('pandemic')

[('epidemic', 0.6707203984260559),
 ('academic', 0.6002153754234314),
 ('coronavirus', 0.5249733328819275),
 ('covid19', 0.5056285262107849),
 ('chrisanderson', 0.5018191933631897),
 ('dictatorship', 0.4976133406162262),
 ('anderson', 0.4881579279899597),
 ('dictator', 0.467024028301239),
 ('outbreak', 0.45039117336273193),
 ('opioid', 0.44782328605651855)]

### Use the trained model's doesnt_match method

In [ ]:
model.wv.doesnt_match(['hockey', 'music', 'baseball'])

'music'

In [ ]:
model.wv.doesnt_match(['conversation', 'joke', 'people', 'fighting'])

'fighting'

In [ ]:
# Create functions to find most similar TEDx talks based on a query
def get_phrase_vector(phrase, model):
    """
    Calculates the average word vector for a given phrase.
    """
    words = phrase.lower().split()
    vectors = [model.wv[word] for word in words if word in model.wv]
    if vectors:
        return np.mean(vectors, axis=0)
    else:
        return None

def find_relevant_talks(phrase, model, df, top_n=5):
    """
    Finds TEDx talks relevant to a given phrase based on cosine similarity,
    filtering results with a minimum similarity threshold.
    """
    phrase_vector = get_phrase_vector(phrase, model)
    if phrase_vector is None:
        return "Phrase not found in vocabulary."

    talk_vectors = [get_phrase_vector(" ".join(title), model) for title in df['title'].tolist()]
    talk_vectors = [v if v is not None else np.zeros(model.vector_size) for v in talk_vectors] #handle nulls.

    similarities = cosine_similarity([phrase_vector], talk_vectors)[0]
    relevant_talks = pd.DataFrame({'title': df['title'], 'similarity': similarities})
    relevant_talks = relevant_talks.sort_values(by='similarity', ascending=False).head(top_n)
    return relevant_talks

In [ ]:
# Example Queries
queries = [
    # 🚀 Technology & Innovation
    "The future of artificial intelligence in daily life",
    "How blockchain is changing the world",
    "The rise of quantum computing and its impact",
    "The ethics of artificial intelligence",
    "Can robots have emotions?",
    "How self-driving cars are revolutionizing transportation",

    # 🧠 Psychology & Personal Development
    "The psychology of happiness and success",
    "How to develop a growth mindset",
    "Overcoming fear and self-doubt",
    "The science of productivity and focus",
    "How gratitude changes your brain",
    "The hidden power of introverts",

    # 🌍 Social Issues & Leadership
    "How to be a great leader in the 21st century",
    "The power of storytelling in social change",
    "Rethinking education for the future",
    "Breaking gender stereotypes in the workplace",
    "The impact of climate change on global health",
    "How social media is shaping modern society",

    # 🔬 Science & Health
    "The future of genetic engineering",
    "How sleep affects mental health",
    "The role of gut bacteria in overall health",
    "The neuroscience behind decision-making",
    "Can we cure aging?",
    "The surprising science of meditation",

    # 🎭 Creativity & Arts
    "The art of public speaking and persuasion",
    "How music influences human emotions",
    "The science behind viral videos",
    "Why creativity is the key to innovation",
    "The psychology of great storytelling",
    "The hidden messages in our favorite movies",

    # 🌎 Global Issues & Society
    "The future of sustainable energy",
    "Can we solve the global water crisis?",
    "The impact of automation on jobs",
    "How universal basic income could change the world",
    "What history can teach us about pandemics",
    "The ethical dilemma of AI in warfare",

    # 🏆 Motivation & Success
    "The secret habits of highly successful people",
    "How failure leads to success",
    "Why you should embrace discomfort",
    "The importance of resilience in life",
    "The power of small daily habits",
    "What makes a great mentor?",

    # 💡 Thought-Provoking & Philosophical Topics
    "What does it mean to live a meaningful life?",
    "Is time just an illusion?",
    "How ancient philosophy can improve modern life",
    "The paradox of choice: why more options make us unhappy",
    "Do we have free will, or is everything predetermined?",
    "Can science explain consciousness?",
]

In [ ]:
for query in queries:
    relevant_talks = find_relevant_talks(query, model, df)
    print(f"\nTop TEDx talks relevant to '{query}':\n{relevant_talks}")


Top TEDx talks relevant to 'The future of artificial intelligence in daily life':
                                title  similarity
2756            Who am I? Think again    0.027886
3626             Your brain on improv    0.027358
3969             Biomimicry in action    0.024274
4114  My library of human imagination    0.021056
4205                      Brain magic    0.015708

Top TEDx talks relevant to 'How blockchain is changing the world':
                      title  similarity
4202            On humanity    0.019094
2756  Who am I? Think again    0.012345
3626   Your brain on improv    0.000828
977     Rethinking thinking   -0.001671
3969   Biomimicry in action   -0.005473

Top TEDx talks relevant to 'The rise of quantum computing and its impact':
                             title  similarity
3245     A 40-year plan for energy    0.088434
3337  Back to the future (of 1994)    0.069692
3849           Innovating to zero!    0.067427
1750   Are you a giver or a taker?    0.06677